<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Deep Learning for MNIST Classification — Implementation</b></h1>
</div>

## Setup — Environment and Configuration


In [1]:

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from mnist import MNIST
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
BATCH_SIZE = 512
HIDDEN_LAYER_WIDTHS = [128, 256, 512]
EPOCHS = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
N_CLASSES = 10

IMAGE_SIZE = 28

INPUT_DIM = IMAGE_SIZE * IMAGE_SIZE
OUTPUT_DIR = Path("../outputs/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

PyTorch version: 2.14.0+cu130
Device: cpu


## 1. Validate MNIST Data and Output Paths


In [ ]:

DATA_DIR = Path("../data")

discovered_mnist_files = {
    path.name: path
    for path in DATA_DIR.glob("*-ubyte")
    if path.is_file()
}
REQUIRED_MNIST_FILES = {
    "train-images-idx3-ubyte",
    "train-labels-idx1-ubyte",
    "t10k-images-idx3-ubyte",
    "t10k-labels-idx1-ubyte",
}

missing_mnist_files = sorted(
    REQUIRED_MNIST_FILES - set(discovered_mnist_files)
)

if missing_mnist_files:
    raise FileNotFoundError(
        "Missing MNIST IDX files: " + ", ".join(missing_mnist_files)
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MNIST data directory : {DATA_DIR}")
print(f"IDX files discovered : {len(discovered_mnist_files)}")
print("Required containers  : 4/4")
print(f"Figures directory    : {OUTPUT_DIR}")

## 2. Build the MNIST Dataset and Mini-Batch Pipeline


In [ ]:

class MNISTDataset(Dataset):

    image_size = IMAGE_SIZE

    def __init__(self, partition, mnist_directory):
        """Initialize the object."""

        if partition not in ("training", "testing"):
            raise ValueError(
                f"{partition!r} is not a valid partition."
            )

        mnist = MNIST(str(mnist_directory))
        parser = getattr(mnist, f"load_{partition}")

        self.images, self.labels = parser()
        self.nimages = len(self.images)

    def __len__(self):
        """Return the number of samples."""
        return self.nimages

    def __getitem__(self, index):
        """Return one sample by index."""

        image = (
            np.asarray(
                self.images[index],
                dtype=np.float32,
            )
            / 255.0
        )

        label = int(self.labels[index])

        return {
            "image": image,
            "label": label,
        }

    @staticmethod
    def collate_fn(data_batch):
        """Combine samples into one training batch."""

        images = np.stack(
            [item["image"] for item in data_batch],
            axis=0,
        ).astype(np.float32)
        labels = np.asarray(
            [item["label"] for item in data_batch],
            dtype=np.int64,
        )

        return {
            "image": images,
            "label": labels,
        }
def build_mnist_dataset_and_loader(
    batch_size,
    partition,
    dataset_directory,
):
    """Build mnist dataset and loader."""
    dataset = MNISTDataset(
        partition=partition,
        mnist_directory=dataset_directory,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,

        shuffle=(partition == "training"),

        num_workers=0,
        collate_fn=dataset.collate_fn,
    )

    return dataset, loader

print("MNIST dataset and DataLoader pipeline defined.")

## 3. Load and Validate Training and Testing Data


In [4]:

train_dataset, training_data_loader = build_mnist_dataset_and_loader(
    batch_size=BATCH_SIZE,
    partition="training",
    dataset_directory=DATA_DIR,
)

test_dataset, test_data_loader = build_mnist_dataset_and_loader(
    batch_size=BATCH_SIZE,
    partition="testing",
    dataset_directory=DATA_DIR,
)
if len(train_dataset) != 60_000:
    raise ValueError(
        f"Expected 60,000 training images; found {len(train_dataset):,}."
    )
if len(test_dataset) != 10_000:
    raise ValueError(
        f"Expected 10,000 test images; found {len(test_dataset):,}."
    )

sample = train_dataset[0]
if sample["image"].shape != (INPUT_DIM,):
    raise ValueError(
        f"Expected flattened image shape {(INPUT_DIM,)}; "
        f"found {sample['image'].shape}."
    )
if not (0 <= sample["label"] < N_CLASSES):
    raise ValueError("MNIST label is outside the expected range 0–9.")
if (
    sample["image"].min() < 0.0
    or sample["image"].max() > 1.0
):
    raise ValueError("MNIST pixels are not normalized to [0, 1].")

print(f"Training images: {len(train_dataset):,}")
print(f"Testing images: {len(test_dataset):,}")
print(f"Flattened image shape: {sample['image'].shape}")
print(f"Pixel range: [{sample['image'].min():.3f}, {sample['image'].max():.3f}]")

Training images: 60,000
Testing images: 10,000
Flattened image shape: (784,)
Pixel range: [0.000, 1.000]


## 4. Visualize Representative MNIST Samples


In [5]:

sample_indices = np.arange(12)

fig, axes = plt.subplots(
    3,
    4,
    figsize=(8, 6),
)

axes = axes.reshape(-1)

for ax, index in zip(
    axes,
    sample_indices,
):
    item = train_dataset[index]

    ax.imshow(
        item["image"].reshape(
            IMAGE_SIZE,
            IMAGE_SIZE,
        ),
        cmap="gray",
    )

    ax.set_title(
        f"Label: {item['label']}"
    )
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "mnist_sample_batch.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 5. Define the One-Hidden-Layer MLP Classifier


In [6]:

class MNISTClassifier(nn.Module):

    def __init__(
        self,
        hidden_layer_width,
        nclasses=N_CLASSES,
        imsize=IMAGE_SIZE,
        use_batch_norm=True,
    ):
        """Initialize the object."""
        super().__init__()

        layers = [
            nn.Linear(
                imsize * imsize,
                hidden_layer_width,
            ),
        ]

        if use_batch_norm:
            layers.append(
                nn.BatchNorm1d(
                    hidden_layer_width
                )
            )
        layers.extend(
            [
                nn.ReLU(),
                nn.Linear(
                    hidden_layer_width,
                    nclasses,
                ),
            ]
        )

        self.net = nn.Sequential(*layers)

        self.loss_fn = nn.functional.cross_entropy

    def forward(self, data_dict):
        """Forward."""
        logits = self.net(
            data_dict["image"]
        )
        if self.training:
            loss = self.loss_fn(
                logits,
                data_dict["label"],
            )

            return {
                "loss": loss,
                "logits": logits,
            }

        predicted_class, confidence = (
            self.decode_prediction(logits)
        )

        return {
            "cls": predicted_class,
            "prob": confidence,
            "logits": logits,
        }

    @staticmethod
    def decode_prediction(logits):
        """Decode prediction."""

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        confidence, predicted_class = torch.max(
            probabilities,
            dim=1,
        )

        return predicted_class, confidence

print("MNISTClassifier defined.")

MNISTClassifier defined.


## 6. Verify the Model Architecture and Forward Pass


In [7]:

sample_classifier = MNISTClassifier(
    hidden_layer_width=256,
).to(device)

print(sample_classifier)
sample_batch = next(
    iter(training_data_loader)
)

sample_batch = {
    "image": torch.from_numpy(
        sample_batch["image"]
    ).to(device),
    "label": torch.from_numpy(
        sample_batch["label"]
    ).to(device),
}

sample_classifier.train()

sample_logits = sample_classifier(
    sample_batch
)
if not torch.isfinite(
    sample_logits["loss"]
):
    raise ValueError(
        "Example forward pass produced a non-finite loss."
    )

expected_shape = (
    sample_batch["image"].shape[0],
    N_CLASSES,
)
if tuple(
    sample_logits["logits"].shape
) != expected_shape:
    raise ValueError(
        "Unexpected logit shape: "
        f"{tuple(sample_logits['logits'].shape)}"
    )

print(
    f"Example training loss: "
    f"{sample_logits['loss'].item():.4f}"
)
print(
    f"Logit shape: "
    f"{tuple(sample_logits['logits'].shape)}"
)

MNISTClassifier(
  (net): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)
Example training loss: 2.3383
Logit shape: (512, 10)


## 7. Define Training and Evaluation Utilities


In [8]:

def prepare_training_batch(
    batch,
    compute_device,
):
    """Prepare training batch."""
    return {
        "image": torch.from_numpy(
            batch["image"]
        ).to(compute_device),
        "label": torch.from_numpy(
            batch["label"]
        ).to(compute_device),
    }
def train_classifier(
    model,
    training_data_loader,
    optimizer,
    epochs,
    compute_device,
):
    """Train classifier."""

    epoch_loss_history = []

    for epoch in range(epochs):

        model.train()
        accumulated_loss = 0.0

        processed_sample_count = 0

        progress = tqdm(
            training_data_loader,
            desc=f"Epoch {epoch + 1}/{epochs}",
            leave=False,
        )

        for batch in progress:
            batch = prepare_training_batch(
                batch,
                compute_device,
            )
            optimizer.zero_grad()

            output = model(batch)
            loss = output["loss"]
            loss.backward()

            optimizer.step()
            batch_sample_count = (
                batch["image"].shape[0]
            )

            accumulated_loss += (
                loss.item()
                * batch_sample_count
            )

            processed_sample_count += (
                batch_sample_count
            )
        epoch_mean_loss = (
            accumulated_loss
            / processed_sample_count
        )

        epoch_loss_history.append(
            epoch_mean_loss
        )

    return epoch_loss_history
@torch.no_grad()
def evaluate_classifier(
    model,
    data_loader,
    compute_device,
):
    """Evaluate classifier."""

    # Evaluation mode disables training-only behavior so reported metrics are reproducible.
    model.eval()

    all_classes = []

    all_probabilities = []

    all_labels = []

    all_logits = []

    for batch in data_loader:
        batch = prepare_training_batch(
            batch,
            compute_device,
        )

        output = model(batch)

        all_classes.append(
            output["cls"].cpu()
        )
        all_probabilities.append(
            output["prob"].cpu()
        )
        all_labels.append(
            batch["label"].cpu()
        )
        all_logits.append(
            output["logits"].cpu()
        )

    classes = torch.cat(
        all_classes
    )

    probabilities = torch.cat(
        all_probabilities
    )

    labels = torch.cat(
        all_labels
    )

    logits = torch.cat(
        all_logits
    )
    accuracy = (
        classes == labels
    ).float().mean().item()

    return {
        "cls": classes,
        "prob": probabilities,
        "label": labels,
        "logits": logits,
        "accuracy": accuracy,
    }

print("Training and evaluation utilities defined.")

Training and evaluation utilities defined.


## 8. Train the 128-, 256-, and 512-Neuron Models


In [9]:

models = {}
training_histories = {}

model_evaluation_results = {}

for hidden_layer_width in HIDDEN_LAYER_WIDTHS:
    print(
        f"\nTraining model: "
        f"{INPUT_DIM} → {hidden_layer_width} → {N_CLASSES}"
    )

    torch.manual_seed(SEED)

    model = MNISTClassifier(
        hidden_layer_width=hidden_layer_width,
    ).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    losses = train_classifier(
        model=model,
        training_data_loader=training_data_loader,
        optimizer=optimizer,
        epochs=EPOCHS,
        compute_device=device,
    )

    evaluation_result = evaluate_classifier(
        model=model,
        data_loader=test_data_loader,
        compute_device=device,
    )

    models[hidden_layer_width] = model
    training_histories[hidden_layer_width] = losses
    model_evaluation_results[hidden_layer_width] = evaluation_result

    print(
        f"Final training loss: "
        f"{losses[-1]:.4f}"
    )
    print(
        f"Test accuracy: "
        f"{evaluation_result['accuracy']:.4f}"
    )


Training model: 784 → 128 → 10


Final training loss: 0.0795
Test accuracy: 0.9718

Training model: 784 → 256 → 10


Final training loss: 0.0743
Test accuracy: 0.9718

Training model: 784 → 512 → 10


Final training loss: 0.0717
Test accuracy: 0.9726


## 9. Compare Training Loss and Test Accuracy


In [ ]:

fig, ax = plt.subplots(
    figsize=(9, 5)
)

for hidden_layer_width in HIDDEN_LAYER_WIDTHS:
    losses = training_histories[
        hidden_layer_width
    ]

    ax.plot(
        range(1, EPOCHS + 1),
        losses,
        marker="o",
        label=(
            f"{hidden_layer_width} hidden neurons"
        ),
    )

ax.set_title(
    "Training Loss by Hidden-Layer Size"
)
ax.set_xlabel("Epoch")
ax.set_ylabel(
    "Mean Cross-Entropy Loss"
)
ax.set_xticks(
    range(1, EPOCHS + 1)
)
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "training_loss_comparison.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
REFERENCE_REPORTED_LOSSES = {
    128: 0.0753,
    256: 0.0344,
    512: 0.0394,
}

print(
    f"{'Hidden':>8} "
    f"{'Final loss':>12} "
    f"{'Test accuracy':>15} "
    f"{'Reference loss':>15}"
)
print("-" * 55)

for hidden_layer_width in HIDDEN_LAYER_WIDTHS:

    final_loss = (
        training_histories[
            hidden_layer_width
        ][-1]
    )
    test_accuracy = (
        model_evaluation_results[
            hidden_layer_width
        ]["accuracy"]
    )

    print(
        f"{hidden_layer_width:>8d} "
        f"{final_loss:>12.4f} "
        f"{test_accuracy:>15.4f} "
        f"{REFERENCE_REPORTED_LOSSES[hidden_layer_width]:>15.4f}"
    )

## 10. Visualize Predictions and Confidence Scores


In [ ]:

def plot_classifier_predictions(
    model,
    dataset,
    hidden_layer_width,
    output_path,
    n_images=15,
):
    """Plot classifier predictions."""

    # Evaluation mode disables training-only behavior so reported metrics are reproducible.
    model.eval()
    images = np.stack(
        [
            dataset[index]["image"]
            for index in range(n_images)
        ]
    ).astype(np.float32)
    labels = np.asarray(
        [
            dataset[index]["label"]
            for index in range(n_images)
        ],
        dtype=np.int64,
    )

    batch = {
        "image": torch.from_numpy(
            images
        ).to(device),
        "label": torch.from_numpy(
            labels
        ).to(device),
    }

    with torch.no_grad():
        output = model(batch)

    probabilities = torch.softmax(
        output["logits"],
        dim=1,
    ).cpu().numpy()

    fig = plt.figure(
        figsize=(12, 10)
    )

    for index in range(n_images):
        image_axis = plt.subplot(
            5,
            6,
            2 * index + 1,
        )

        image_axis.imshow(
            images[index].reshape(
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),
            cmap="gray",
        )

        predicted_label = int(
            np.argmax(
                probabilities[index]
            )
        )

        confidence = float(
            np.max(
                probabilities[index]
            )
        )

        true_label = int(
            labels[index]
        )

        image_axis.set_xlabel(
            f"Pred {predicted_label} "
            f"({confidence:.0%})\n"
            f"True {true_label}"
        )

        image_axis.set_xticks([])
        image_axis.set_yticks([])

        value_axis = plt.subplot(
            5,
            6,
            2 * index + 2,
        )

        value_axis.bar(
            range(N_CLASSES),
            probabilities[index],
        )

        value_axis.set_ylim(
            0,
            1,
        )
        value_axis.set_xticks(
            range(N_CLASSES)
        )
        value_axis.set_yticks([])

    fig.suptitle(
        f"MNIST Predictions — "
        f"{hidden_layer_width} Hidden Neurons",
        y=1.01,
    )

    plt.tight_layout()

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
for hidden_layer_width in HIDDEN_LAYER_WIDTHS:
    plot_classifier_predictions(
        model=models[hidden_layer_width],
        dataset=test_dataset,
        hidden_layer_width=hidden_layer_width,
        output_path=(
            OUTPUT_DIR
            / f"mnist_predictions_{hidden_layer_width}.png"
        ),
    )

## 11. Compute Confidence-Threshold Precision, Recall, and Accepted Accuracy


In [12]:

def compute_confidence_threshold_curves(
    evaluation_result,

number_of_thresholds=100,
):
    """Compute confidence threshold curves."""
    predicted_classes = (
        evaluation_result["cls"].numpy()
    )

    confidence = (
        evaluation_result["prob"].numpy()
    )

    labels = (
        evaluation_result["label"].numpy()
    )

    correct = (
        predicted_classes
        == labels
    )
    # Threshold sweeping exposes the confidence-versus-coverage trade-off beyond raw accuracy.
    thresholds = np.linspace(
        0.0,
        0.99,
        number_of_thresholds,
    )

    precision_values = []
    recall_values = []

    accepted_accuracy_values = []

    total_correct = np.sum(
        correct
    )

    for threshold in thresholds:

        accepted = (
            confidence
            > threshold
        )

        true_positive = np.sum(
            accepted & correct
        )

        false_positive = np.sum(
            accepted & ~correct
        )

        false_negative = np.sum(
            ~accepted & correct
        )

        precision_denominator = (
            true_positive
            + false_positive
        )

        recall_denominator = (
            true_positive
            + false_negative
        )
        if precision_denominator > 0:
            precision = true_positive / precision_denominator
        else:
            precision = np.nan
        if recall_denominator > 0:
            recall = true_positive / recall_denominator
        else:
            recall = 0.0
        accepted_accuracy = (
            true_positive
            / len(labels)
        )

        precision_values.append(
            precision
        )
        recall_values.append(
            recall
        )
        accepted_accuracy_values.append(
            accepted_accuracy
        )

    precision_values = np.asarray(
        precision_values
    )
    recall_values = np.asarray(
        recall_values
    )
    accepted_accuracy_values = np.asarray(
        accepted_accuracy_values
    )
    order = np.argsort(
        recall_values
    )

    return {
        "thresholds": thresholds[order],
        "precision": precision_values[order],
        "recall": recall_values[order],
        "accepted_accuracy": accepted_accuracy_values[order],
        "total_correct": int(total_correct),
    }

print("Confidence-threshold evaluation utility defined.")

Confidence-threshold evaluation utility defined.


## 12. Analyze the Best Model and Save Evaluation Curves


In [13]:

best_hidden_size = min(
    HIDDEN_LAYER_WIDTHS,
    key=lambda size: (
        training_histories[
            size
        ][-1]
    ),
)

best_evaluation = (
    model_evaluation_results[
        best_hidden_size
    ]
)
confidence_curves = (
    compute_confidence_threshold_curves(
        best_evaluation,

number_of_thresholds=100,
    )
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4.5),
)

axes[0].plot(
    confidence_curves["recall"],
    confidence_curves["precision"],
)
axes[0].set_title(
    "Precision–Recall Curve"
)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].grid(alpha=0.25)

axes[1].plot(
    confidence_curves["recall"],
    confidence_curves[
        "accepted_accuracy"
    ],
)
axes[1].set_title(
    "Accepted Accuracy–Recall Curve"
)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel(
    "Accepted correct / all test samples"
)
axes[1].grid(alpha=0.25)

fig.suptitle(
    f"Confidence-Threshold Evaluation — "
    f"{best_hidden_size} Hidden Neurons"
)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "pr_accuracy_curve.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    f"Selected hidden size: "
    f"{best_hidden_size}"
)
print(
    f"Final training loss: "
    f"{training_histories[best_hidden_size][-1]:.4f}"
)
print(
    f"Test accuracy: "
    f"{best_evaluation['accuracy']:.4f}"
)

Selected hidden size: 512
Final training loss: 0.0717
Test accuracy: 0.9726


## 13. Run Numerical and Output-file Validation Checks


In [ ]:

if len(train_dataset) != 60_000:
    raise ValueError(
        "Training dataset size changed unexpectedly."
    )
if len(test_dataset) != 10_000:
    raise ValueError(
        "Test dataset size changed unexpectedly."
    )
if set(models) != set(HIDDEN_LAYER_WIDTHS):
    raise ValueError(
        "Not all required MLP configurations were trained."
    )

for hidden_layer_width in HIDDEN_LAYER_WIDTHS:

    losses = np.asarray(
        training_histories[
            hidden_layer_width
        ],
        dtype=float,
    )
    if losses.shape != (EPOCHS,):
        raise ValueError(
            f"Unexpected loss-history length for hidden size {hidden_layer_width}."
        )
    if not np.all(
        np.isfinite(losses)
    ):
        raise ValueError(
            f"Non-finite training loss for hidden size {hidden_layer_width}."
        )

    evaluation_result = (
        model_evaluation_results[
            hidden_layer_width
        ]
    )
    if not (
        0.0
        <= evaluation_result["accuracy"]
        <= 1.0
    ):
        raise ValueError(
            f"Invalid test accuracy for hidden size {hidden_layer_width}."
        )
    if (
        evaluation_result["cls"].shape[0]
        != len(test_dataset)
    ):
        raise ValueError(
            f"Incomplete test predictions for hidden size {hidden_layer_width}."
        )

    probabilities = (
        evaluation_result["prob"].numpy()
    )
    if not np.all(
        np.isfinite(probabilities)
    ):
        raise ValueError(
            f"Non-finite confidence values for hidden size {hidden_layer_width}."
        )
    if (
        np.any(probabilities < 0.0)
        or np.any(probabilities > 1.0)
    ):
        raise ValueError(
            f"Confidence values outside [0, 1] for hidden size {hidden_layer_width}."
        )
REQUIRED_OUTPUTS = [
    "mnist_sample_batch.png",
    "training_loss_comparison.png",
    "mnist_predictions_128.png",
    "mnist_predictions_256.png",
    "mnist_predictions_512.png",
    "pr_accuracy_curve.png",
]
missing_outputs = [
    name
    for name in REQUIRED_OUTPUTS
    if not (
        OUTPUT_DIR / name
    ).exists()
]
if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: "
        + ", ".join(
            missing_outputs
        )
    )

print(
    "All Deep Learning validation checks passed."
)